In [ ]:
#This file is just used to deal with data used for Bert classifier
#Data use:
#1.GPT007/Pirate_speak
#2.Random English Sentences（Kaggle）
#3.KafeisM/pirate-speak-dataset
from datasets import load_dataset

ds = load_dataset("GPT007/Pirate_speak")


import json
from pathlib import Path
from datasets import load_dataset

# ---------------------------
# 1. file paths
# ---------------------------
pirate_json_path = Path("pirate_style_transfer_dataset.json")
normal_txt_path = Path("Random English Sentences.txt")
output_path = Path("combined_pirate_dataset.json")

combined_data = []

# ---------------------------
# 2. load pirate data from Hugging Face
#    GPT007/Pirate_speak
# ---------------------------
ds = load_dataset("GPT007/Pirate_speak")

print("Hugging Face splits:", ds)

# inspect one row first if needed:
# print(ds["train"][0])

for split_name in ds.keys():
    for row in ds[split_name]:
        # try common possible field names
        text = None
        for key in ["text", "sentence", "pirate", "output", "response", "content"]:
            if key in row and row[key]:
                text = str(row[key]).strip()
                break

        # if nothing found, fall back to first string value
        if text is None:
            for value in row.values():
                if isinstance(value, str) and value.strip():
                    text = value.strip()
                    break

        if text:
            combined_data.append({
                "sentence": text,
                "label": 1
            })

# ---------------------------
# 3. load pirate data from local json
#    pirate_style_transfer_dataset.json
# ---------------------------
with open(pirate_json_path, "r", encoding="utf-8") as f:
    pirate_json_data = json.load(f)

# handle different possible json formats
if isinstance(pirate_json_data, list):
    for item in pirate_json_data:
        if isinstance(item, dict):
            # likely fields: pirate / english / input / output / transformed_text
            pirate_text = None
            for key in ["pirate", "output", "pirate_text", "transformed_text", "text"]:
                if key in item and item[key]:
                    pirate_text = str(item[key]).strip()
                    break

            if pirate_text:
                combined_data.append({
                    "sentence": pirate_text,
                    "label": 1
                })

elif isinstance(pirate_json_data, dict):
    # if wrapped in a key like {"data": [...]}
    for possible_list_key in ["data", "examples", "items", "records"]:
        if possible_list_key in pirate_json_data and isinstance(pirate_json_data[possible_list_key], list):
            for item in pirate_json_data[possible_list_key]:
                if isinstance(item, dict):
                    pirate_text = None
                    for key in ["pirate", "output", "pirate_text", "transformed_text", "text"]:
                        if key in item and item[key]:
                            pirate_text = str(item[key]).strip()
                            break
                    if pirate_text:
                        combined_data.append({
                            "sentence": pirate_text,
                            "label": 1
                        })
            break

# ---------------------------
# 4. load non-pirate data from txt
# ---------------------------
with open(normal_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        sentence = line.strip()
        if sentence:
            combined_data.append({
                "sentence": sentence,
                "label": 0
            })

# ---------------------------
# 5. optional: remove duplicates
# ---------------------------
seen = set()
deduped_data = []

for item in combined_data:
    key = (item["sentence"], item["label"])
    if key not in seen:
        seen.add(key)
        deduped_data.append(item)

# ---------------------------
# 6. save to json
# ---------------------------
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(deduped_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(deduped_data)} rows to {output_path}")

/Users/yihengli/Documents/python/final_project/tinystories_llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 100/100 [00:00<00:00, 10816.75 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 100
    })
})


In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")
for split, subset in ds.items():
    print(split, len(subset))

/Users/yihengli/Documents/python/final_project/tinystories_llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating valid split: 100%|██████████| 4994/4994 [00:00<00:00, 208676.83 examples/s]

train 11994
valid 4994
